# PRISM: batch extraction with an open-weight model (Kaggle GPU)

1. Locally: `python -m cli batch-export --system mistral-7b` (or `llama-3.1-8b`) writes `backend/data/batch/<system>.requests.jsonl`.
2. Upload that file as a Kaggle dataset and add it to this notebook.
3. Settings: Accelerator **GPU T4 x2**, Internet **on**. For gated models (Llama, Mistral) accept the licence on Hugging Face and add a Kaggle secret **HF_TOKEN**.
4. Run all cells; download `<system>.responses.jsonl` from the Output tab.
5. Locally: `python -m cli batch-import <system>.responses.jsonl`, then `cli experiment` / `cli eval-run` for the system read from the cache.

In [ ]:
!pip install -q vllm

In [ ]:
import glob, json, os
src = glob.glob('/kaggle/input/*/*.requests.jsonl')
assert len(src) == 1, f'add exactly one requests file as a dataset, found {src}'
lines = [json.loads(l) for l in open(src[0], encoding='utf-8') if l.strip()]
header, reqs = lines[0], lines[1:]
print(header, len(reqs), 'requests')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no HF_TOKEN secret (fine for non-gated models):', e)

In [ ]:
import torch, vllm
from vllm import LLM, SamplingParams
from huggingface_hub import model_info
model = header['model']
revision = model_info(model, token=os.environ.get('HF_TOKEN')).sha
llm = LLM(model=model, revision=revision, dtype='half', max_model_len=8192,
          tensor_parallel_size=max(1, torch.cuda.device_count()), gpu_memory_utilization=0.90)
version = f'{model}@{revision[:12]} vllm {vllm.__version__}'
print(version)

In [ ]:
def params(r):
    kw = dict(temperature=r['temperature'], seed=r['seed'], max_tokens=header['max_tokens'])
    try:   # JSON-constrained decoding where this vLLM version supports it
        from vllm.sampling_params import GuidedDecodingParams
        return SamplingParams(**kw, guided_decoding=GuidedDecodingParams(json_object=True))
    except Exception:
        return SamplingParams(**kw)

out_path = f"/kaggle/working/{header['system']}.responses.jsonl"
with open(out_path, 'w', encoding='utf-8') as fh:
    fh.write(json.dumps({'system': header['system'], 'model_version': version}) + '\n')
    B = 64
    for i in range(0, len(reqs), B):
        chunk = reqs[i:i + B]
        outs = llm.chat([[{'role': 'user', 'content': r['prompt']}] for r in chunk],
                        [params(r) for r in chunk], use_tqdm=False)
        for r, o in zip(chunk, outs):
            fh.write(json.dumps({'key': r['key'], 'text': o.outputs[0].text,
                                 'model_version': version}, ensure_ascii=False) + '\n')
        fh.flush()
        print(min(i + B, len(reqs)), '/', len(reqs))
print('wrote', out_path)